In [1]:
import os
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import requests
from IPython.display import Markdown, display
from openai import OpenAI
import json
from IPython.display import Markdown, display, update_display

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')



if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [ ]:

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [4]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [5]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Welcome — nice to meet you. I’m ChatGPT, and I can help with answers, writing, learning, planning, coding, and more.\n\nWhat would you like to do today? Here are a few ideas:\n- Ask me a question on any topic\n- Learn something new or dive into a topic you’re curious about\n- Get help with writing (emails, essays, stories, resumes)\n- Work on a homework or project task\n- Practice a language or have a casual chat\n- Plan something (trip, event, study schedule)\n- Quick fun: a joke, a short story, or trivia\n\nTell me a bit about your interests or what you’re hoping to accomplish, and I’ll tailor my help. What should we start with?'

In [6]:
sai = fetch_website_contents("https://cnn.com")
print(sai)

Breaking News, Latest News and Videos | CNN

CNN values your feedback
1. How relevant is this ad to you?
2. Did you encounter any technical issues?
Video player was slow to load content
Video content never loaded
Ad froze or did not finish loading
Video content did not start after ad
Audio on ad was too loud
Other issues
Ad never loaded
Ad prevented/slowed the page from loading
Content moved around while ad loaded
Ad was repetitive to ads I've seen previously
Other issues
Cancel
Submit
Thank You!
Your effort and contribution in providing this feedback is much
                                        appreciated.
Close
Ad Feedback
Close icon
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Winter Olympics 2026
Ukraine-Russia War
Israel-Hamas War
Games
More
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Winter Olympics 2026
Ukraine-Russia War
Israel-Hamas War
Games
Watch
Listen
Live TV
Subscribe
Sign i

In [7]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [8]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

In [10]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [11]:
messages_for(sai)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': "\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nBreaking News, Latest News and Videos | CNN\n\nCNN values your feedback\n1. How relevant is this ad to you?\n2. Did you encounter any technical issues?\nVideo player was slow to load content\nVideo content never loaded\nAd froze or did not finish loading\nVideo content did not start after ad\nAudio on ad was too loud\nOther issues\nAd never loaded\nAd prevented/slowed the page from loading\nContent moved around while ad loaded\nAd was repetitive to ads I've seen previously\nOther issues\nCancel\nSubmit\nThank Yo

In [12]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [13]:
summarize("https://en.wikipedia.org/wiki/Sundar_Pichai")

'# Sundar Pichai - Wikipedia, aka "The Google Overlord"\n\nThis website is basically the Wikipedia shrine to Sundar Pichai, the guy who runs Alphabet Inc. and Google since 2015 (no biggie). Born in 1972 in Madurai, India, he went from IIT Kharagpur to Stanford, then nabbed an MBA from the University of Pennsylvania. Now, he\'s the CEO, a board member, and has even rubbed elbows with Magic Leap.\n\nHighlights include his humble beginnings, career ladder climbing, awards like India\'s Padma Bhushan in 2022, and some family deets (married to Anjali with two kids). If you came here expecting hot news or juicy scandals—sorry, this one’s a straight-laced bio, with no breaking headlines or surprise announcements. Just a neat package of a tech titan\'s life story.\n\nIn short: Sundar = Indian tech genius + American dream + Google overlord status.'

In [14]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [15]:
display_summary("https://en.wikipedia.org/wiki/Sundar_Pichai")

Ah, Sundar Pichai's Wikipedia page—the ultimate tribute to the tech world's favorite CEO who took over Google and Alphabet like a boss since 2015. Born in India, armed with degrees from IIT, Stanford, and UPenn (basically a diploma buffet), and now ruling the Silicon Valley kingdom. The page covers his life journey from Madurai to modern tech royalty, cups an MBA, a Padma Bhushan award for good measure, and family life with a spouse named Anjali and two kids. No scandal, no drama—just a neat, encyclopedic walk through the life of the guy who probably knows more about your internet habits than your closest friends. No breaking news or spicy announcements here, just classic biography served Wikipedia style.

In [16]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [17]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-D5lajMPWmQ9DkJLWjHHcEdJaetPdV',
 'object': 'chat.completion',
 'created': 1770266953,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Wombats poop cubes. The cube-shaped stools help them mark territory and prevent rolling away on the Australian ground, and they’re formed by the way their intestines process the material slowly.',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 1266,
  'total_tokens': 1277,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 1216,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [18]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Wombats poop cubes. The cube-shaped stools help them mark territory and prevent rolling away on the Australian ground, and they’re formed by the way their intestines process the material slowly.'

In [19]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Fun fact: A day on Venus is longer than its year—you’d have a longer day (about 243 Earth days) than a year (about 225 Earth days) there. Plus, Venus rotates in the opposite direction, so the Sun rises in the west.'

In [20]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [21]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Did you know that a group of flamingos is called a "flamboyance"?'

In [22]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [23]:
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Here's one:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible!\n\nHoney's unique properties make it virtually indestructible. It has a low water content and is saturated with beeswax, which acts as a natural preservative. This means that as long as the honey remains sealed and dry, it will remain stable and safe to eat for eternity!"

In [24]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [25]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [26]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [27]:
encoding.decode([326])

' and'

In [28]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm sai!"}
    ]

In [29]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Hello Sai! How can I assist you today?'

In [30]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [31]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

"I don't have access to personal information about users unless you share it with me during our conversation. How can I assist you today?"

In [32]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm sai!"},
    {"role": "assistant", "content": "Hi sai! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [33]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Your name is sai! How can I help you today, sai?'

In [34]:
links = fetch_website_links("https://cnn.com")
links

['https://edition.cnn.com',
 'https://edition.cnn.com/us',
 'https://edition.cnn.com/world',
 'https://edition.cnn.com/politics',
 'https://edition.cnn.com/business',
 'https://edition.cnn.com/health',
 'https://edition.cnn.com/entertainment',
 'https://edition.cnn.com/style',
 'https://edition.cnn.com/travel',
 'https://edition.cnn.com/sports',
 'https://edition.cnn.com/science',
 'https://edition.cnn.com/climate',
 'https://edition.cnn.com/weather',
 'https://edition.cnn.com/sport/milan-cortina-winter-olympics-2026',
 'https://edition.cnn.com/world/europe/ukraine',
 'https://edition.cnn.com/world/middleeast/israel',
 'https://edition.cnn.com/games',
 'https://edition.cnn.com/us',
 'https://edition.cnn.com/world',
 'https://edition.cnn.com/politics',
 'https://edition.cnn.com/business',
 'https://edition.cnn.com/health',
 'https://edition.cnn.com/entertainment',
 'https://edition.cnn.com/style',
 'https://edition.cnn.com/travel',
 'https://edition.cnn.com/sports',
 'https://edition.cn

In [35]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [36]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [37]:
print(get_links_user_prompt("https://cnn.com"))


Here is the list of links on the website https://cnn.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edition.cnn.com
https://edition.cnn.com/us
https://edition.cnn.com/world
https://edition.cnn.com/politics
https://edition.cnn.com/business
https://edition.cnn.com/health
https://edition.cnn.com/entertainment
https://edition.cnn.com/style
https://edition.cnn.com/travel
https://edition.cnn.com/sports
https://edition.cnn.com/science
https://edition.cnn.com/climate
https://edition.cnn.com/weather
https://edition.cnn.com/sport/milan-cortina-winter-olympics-2026
https://edition.cnn.com/world/europe/ukraine
https://edition.cnn.com/world/middleeast/israel
https://edition.cnn.com/games
https://edition.cnn.com/us
https://edition.cnn.com/world
https://edition.cnn.com/politics
https://edition.cnn.com/busi

In [38]:
MODEL = 'gpt-5-nano'

In [39]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [40]:
select_relevant_links("https://cnn.com")

{'links': [{'type': 'about page', 'url': 'https://edition.cnn.com/about'},
  {'type': 'careers page', 'url': 'https://careers.wbd.com/cnnjobs'},
  {'type': 'company page', 'url': 'https://www.linkedin.com/company/cnn'}]}

In [41]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [42]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 21 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'join page', 'url': 'https://huggingface.co/join'},
  {'type': 'Discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'product page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Facebook', 'url': 'https://huggingface.co/facebook'},
  {'type': 'partner page

In [43]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [44]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
moonshotai/Kimi-K2.5
Updated
about 1 hour ago
•
152k
•
1.68k
zai-org/GLM-OCR
Updated
2 days ago
•
34.3k
•
614
stepfun-ai/Step-3.5-Flash
Updated
2 days ago
•
5.6k
•
429
circlestone-labs/Anima
Updated
4 days ago
•
34.4k
•
408
Qwen/Qwen3-Coder-Next
Updated
1 day ago
•
3.09k
•
391
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.22k
Qwen3-TTS Demo
🎙
1.22k
Transform text into natural-sounding speech with custom voices
Running
431
Demo Playground
⚡
431
Free platform to access multiple AI models
Running
on
Zero
MCP
2k
Z Image Turbo
🖼
2k


In [45]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [46]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [47]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2.5\nUpdated\nabout 1 hour ago\n•\n152k\n•\n1.68k\nzai-org/GLM-OCR\nUpdated\n2 days ago\n•\n34.3k\n•\n614\nstepfun-ai/Step-3.5-Flash\nUpdated\n2 days ago\n•\n5.6k\n•\n429\ncirclestone-labs/Anima\nUpdated\n4 days ago\n•\n34.4k\n•\n408\nQwen/Qwen3-Coder-Next\nUpdated\n1 day ago\n•\n3.09k\n•\n391\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.22k\nQwen3-TTS Demo\n🎙

In [48]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [49]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is **the AI community building the future** — a collaborative platform at the heart of the machine learning ecosystem where engineers, scientists, and innovators come together to create, share, and advance AI models, datasets, and applications. It serves as the premier hub for open-source machine learning collaboration and innovation, empowering users worldwide to learn, build, and grow in an ethical and open AI future.

---

## Key Features

- **Models:** Access and contribute to over **2 million machine learning models** spanning multiple modalities including text, image, video, audio, and 3D.
- **Datasets:** Explore a vast repository of **over 500,000 datasets** to accelerate training and experimentation.
- **Spaces:** Build, deploy, and share AI applications effortlessly using Hugging Face’s hosted Spaces — interactive environments running on powerful hardware.
- **Community:** Join a fast-growing, passionate community focused on collaboration, open-source innovation, and ethical AI development.
- **Enterprise Solutions:** Teams and enterprises can leverage Hugging Face’s advanced platform with enterprise-grade security, access controls, and paid compute solutions tailored for professional use.

---

## What Makes Hugging Face Unique?

- **Collaboration Platform:** Unlimited hosting and collaboration on public models, datasets, and AI apps.
- **Diversified Modalities:** Tools and libraries supporting all data types to cover nearly every AI use case imaginable.
- **Open Source Excellence:** Access to some of the most widely used open source machine learning libraries that power cutting-edge AI research and products.
- **Community-Driven Development:** Users can explore trending models and datasets, share work, build AI profiles, and partake in collective development.

---

## Company Culture

- **Community Centric:** Hugging Face thrives on open collaboration and shared innovation, believing in building AI for the common good.
- **Ethical AI:** Commitment to an ethical, transparent, and responsible AI future.
- **Fast-Paced and Open:** Encourages rapid experimentation, learning, and sharing through open-source frameworks.
- **Inclusive Innovation:** Empowering a new generation of machine learning talent from around the world.

---

## Serving a Wide Audience

- Individual ML engineers and researchers growing their skills and portfolios.
- Enterprises seeking scalable, secure AI infrastructure.
- AI developers creating applications in natural language processing, computer vision, speech, and generative AI.
- Researchers and organizations looking to collaborate openly on AI projects.

---

## Careers at Hugging Face

Join a pioneering team dedicated to shaping the future of AI! Hugging Face offers opportunities for individuals passionate about machine learning, open source, and community-driven innovation. Careers include roles in:

- Research and Development
- Software Engineering
- DevOps & Infrastructure
- Community & Open Source Management
- Enterprise Solutions and Customer Success

If you thrive in a forward-thinking, mission-driven environment and want to contribute to ethical AI development, Hugging Face could be your next great opportunity.

---

## Get Started

Explore thousands of AI models, build your own, or deploy innovative apps with Hugging Face:

- Visit: [huggingface.co](https://huggingface.co)
- Sign up for free to start collaborating
- Discover trending models, datasets, and Spaces
- Scale your projects with enterprise-grade compute and services

---

**Hugging Face** — where the machine learning community creates the future of AI, together.

In [50]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [51]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the leading AI community and collaboration platform that is building the future of machine learning. It serves as a central hub where machine learning practitioners—from engineers and scientists to researchers and developers—can create, discover, share, and collaborate on models, datasets, and AI applications.

Operating an open and ethical AI ecosystem, Hugging Face empowers the next generation in AI to learn, build, and innovate together in an accessible, transparent, and community-driven environment.

---

## What Hugging Face Offers

- **Models:** Explore and contribute to a vast repository of over **2 million machine learning models** covering various AI modalities including text, image, video, audio, and 3D.
- **Datasets:** Access a comprehensive collection of **over 500,000 datasets** curated by the community and contributors worldwide.
- **Spaces:** Engage with live, hosted AI applications and demos built by the community, spanning natural language processing, speech synthesis, image generation, music creation, and beyond.
- **Open Source Stack:** Leverage Hugging Face’s open source tools and libraries to accelerate development and collaboration.
- **Enterprise Solutions:** Benefit from paid compute resources and enterprise-grade platforms designed for secure, scalable AI development and deployment for companies and teams.
- **Community & Collaboration:** Join a vibrant, fast-growing community of ML practitioners contributing to open, ethical AI projects and sharing their work publicly to build their professional profiles.

---

## Company Culture

- **Collaboration-First:** At its core, Hugging Face champions open collaboration, making machine learning accessible and interactive to all.
- **Open & Ethical AI:** Dedicated to promoting transparency, accessibility, and ethical development in AI technologies.
- **Community Empowerment:** Building an inclusive ecosystem where contributions from around the world help push forward innovation.
- **Innovation & Speed:** Providing tools and platforms that allow developers to move faster and build better AI solutions with ease.
- **Learning & Growth:** Encourages sharing knowledge, building portfolios, and growing skills across all levels of AI expertise.

---

## Who Uses Hugging Face?

- **Machine Learning Engineers & Researchers:** Utilize the platform for state-of-the-art models and datasets to accelerate research and deployment.
- **Developers & Data Scientists:** Build new applications, experiment with AI models, and share projects publicly or within teams.
- **Enterprises:** Adopt Hugging Face’s secure, scalable enterprise offerings for internal AI development with fine-grained access control.
- **Educators & Students:** Leverage open resources and community content for learning, experimentation, and collaborative projects.
- **AI Enthusiasts & Innovators:** Access a rich ecosystem of demos, apps, and AI models to inspire creativity and exploration.

---

## Careers at Hugging Face

Hugging Face seeks passionate individuals who want to be at the forefront of AI innovation and collaborate with a world-class community. Opportunities exist in:

- Machine learning research and engineering
- Platform development and software engineering
- Community management and developer relations
- Enterprise solutions and customer success
- Product management and design

Joining Hugging Face means being part of a mission-driven team that values innovation, openness, diverse perspectives, and ethical AI practices.

---

## Get Involved

- **Explore AI Apps and Models:** Start experimenting with thousands of open-source models and curated datasets.
- **Share Your Work:** Upload and host your own models, datasets, and AI applications on the Hugging Face Hub.
- **Join the Community:** Connect with fellow machine learning practitioners, contribute to projects, and build your portfolio.
- **Enterprise & Compute:** Scale your AI projects with Hugging Face’s trusted infrastructure and professional support.

---

**Website:** [huggingface.co](https://huggingface.co)  
**Join the AI revolution and build the future with Hugging Face!**

---

*Colors inspired by the brand identity:*  
Yellow #FFD21E | Orange #FF9D00 | Gray #6B7280